# Training Data Generator - Cost Estimate

## Generating 5,000 Training Examples with GPT-4.1-mini

**Pricing (as of Nov 2024):**
- Input tokens: $0.40 per million tokens
- Output tokens: $1.60 per million tokens

**Estimated cost for 5,000 examples:**

Assuming each request uses:
- ~10,000 input tokens (example report + system prompt)
- ~5,000 output tokens (generated reasoning + verdict)

**Calculation:**
- Input: 5,000 × 10,000 = 50M tokens → 50 × $0.40 = **$20.00**
- Output: 5,000 × 5,000 = 25M tokens → 25 × $1.60 = **$40.00**

**Total Estimated Cost: $60.00** (approximately $40-60 USD depending on actual token usage)

---

**Rate Limiting & Retries:**
- Uses `limiter` package: 60 requests/minute with 120 burst capacity
- Automatic retries via `tenacity`: 3 attempts with exponential backoff (4-15 seconds)
- OpenAI client also configured with max_retries=3
- Estimated time: ~90 minutes for 5,000 examples (due to rate limiting)

In [1]:
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from typing import List, Literal
from enum import Enum
import json
from pathlib import Path
from tqdm.asyncio import tqdm_asyncio
import asyncio
from datetime import datetime
import os
import random
from limiter import Limiter
from tenacity import retry, stop_after_attempt, wait_exponential

client = AsyncOpenAI(
    timeout=60*5,
    max_retries=3,
)

# Rate limiter - adjust based on your OpenAI tier
limiter = Limiter(
    rate=250,
    capacity=120,  # burst capacity
)

In [2]:
class ProceedSignal(str, Enum):
    """Investment decision signal."""
    STRONG_YES = "strong_yes"
    QUESTIONABLE = "questionable"
    STRONG_NO = "strong_no"


class DealVerdict(BaseModel):
    """Structured verdict on a potential PE deal."""
    
    proceed_signal: ProceedSignal = Field(
        description="Overall investment signal: strong_yes (high conviction opportunity), questionable (needs more analysis), or strong_no (pass on deal)"
    )
    
    evidence_pro: str = Field(
        description="Key positive factors supporting investment: competitive advantages, growth drivers, financial strengths, market opportunities",
    )
    
    evidence_contra: str = Field(
        description="Key concerns or risks against investment: competitive threats, financial weaknesses, market headwinds, execution risks",
    )
    
    similar_deals: str = Field(
        description="Comparable past PE deals (fictional examples) in similar sectors/situations that provide context. Format as brief descriptions of buyouts, growth equity, add-on acquisitions, etc."
    )
    
    key_information_requested: str = Field(
        description="Critical additional information, analysis, or due diligence needed to make final investment decision, even if signal is positive"
    )


class TrainingExample(BaseModel):
    """Complete training data example for fine-tuning."""
    
    report: str = Field(
        description="The generated company research report"
    )
    
    reasoning: str = Field(
        description="Conversational, first-person analytical reasoning in a thinking-aloud style. Use phrases like 'Okay, looking at this company...', 'Let me analyze the financials...', 'This company seems to...'. Walk through competition, customers, financials, and growth naturally."
    )
    
    verdict: DealVerdict = Field(
        description="The structured investment verdict with proceed signal and supporting evidence"
    )

In [3]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=15),
    reraise=True,
)
async def generate_training_example(
    example_report_path: str,
    model: str = "gpt-4.1-mini",
    temperature: float = 1.0,
    client: AsyncOpenAI = None,
) -> TrainingExample:
    """
    Generate synthetic training data by creating a FAKE company report and corresponding verdict.
    
    Includes rate limiting via limiter and automatic retries via tenacity.
    Randomly selects a target verdict to ensure balanced distribution.
    """
    
    # Randomly select target verdict for balanced distribution
    target_verdict = random.choice(list(ProceedSignal))
    
    # Read the example report as a template
    example_report_text = Path(example_report_path).read_text()
    
    system_prompt = f"""You are a synthetic training data generator for private equity deal analysis.

Your task is to generate COMPLETELY FICTIONAL training data consisting of:
1. A fictional company research report 
2. An investment verdict on that fake company
3. Analytical reasoning about that verdict

TARGET VERDICT: Your generated example should aim for a "{target_verdict.value}" verdict. Create a company and scenario that naturally leads to this conclusion.

The example report provided is ONLY a loose reference - DO NOT copy its structure rigidly. You must:

REPORT GENERATION:
- Invent a completely new company (different industry, different name, different business model)
- Generate realistic but fake financial data (revenue, margins, growth rates, balance sheet)
- Create fake competitors, customers, and market dynamics
- Make all facts, figures, and dates fictional but internally consistent
- Create a scenario that naturally leads to a "{target_verdict.value}" verdict

REPORT STRUCTURE FLEXIBILITY:
- The example structure and topics are JUST EXAMPLES - adapt freely to fit YOUR company
- Add company-specific sections when relevant (e.g., "Regulatory Environment" for healthcare, "Technology Stack" for SaaS, "Supply Chain" for manufacturing, "Unit Economics" for marketplaces)
- Remove sections that aren't relevant to your company
- Vary the depth and topics covered - sometimes focus more on Competition, other times on Customers or Growth
- GOAL IS DIVERSITY, NOT LENGTH - reports should be tailored to what's relevant for each company

Then, analyze YOUR OWN generated fake report through these lenses:
1. COMPETITION: Market position, competitive dynamics, defensibility
2. CUSTOMERS: Customer quality, retention, concentration risks  
3. FINANCIALS: Growth trajectory, profitability, cash generation, balance sheet
4. GROWTH OPPORTUNITIES: Organic growth, M&A, operational improvements

REASONING STYLE: Write your reasoning in a conversational, thinking-aloud first-person style. Example: "Okay, looking at this company... Let me analyze the financials here. This company seems to have strong margins, but..."

Finally, determine:
- proceed_signal: {target_verdict.value} (as targeted)
- evidence_pro: specific positive factors from YOUR fake report (3-6 points)
- evidence_contra: specific concerns from YOUR fake report (3-6 points)
- similar_deals: fictional PE deals that provide comps (e.g., "Thoma Bravo's $2.1B buyout of ConnectWise in 2019")
- key_information_requested: critical additional diligence needed

Generate diverse, realistic examples. Vary industries, company stages, financial profiles, report structures, and verdict outcomes."""

    user_prompt = f"""Here is an EXAMPLE report (DO NOT copy this structure or topics - adapt freely to fit your company):

{example_report_text}

---

Now generate a COMPLETELY NEW fictional company and deal memo with a structure and sections that fit YOUR company's industry and context. Then provide your reasoning and investment verdict for YOUR generated company."""
    
    async with limiter:
        response = await client.responses.parse(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            text_format=TrainingExample,
        )
    
    result = response.output_parsed
    
    # Save individual example to file

    output_dir = "../data/output/individual"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S-%f")[:-3]  # milliseconds
    output_file = os.path.join(output_dir, f"{timestamp}.json")
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result.model_dump(), f, indent=2, ensure_ascii=True)
    
    return result

In [4]:
async def generate_batch(
    example_report_path: str,
    num_examples: int = 10,
    client: AsyncOpenAI = None,
    temperature: float = 1.5,
) -> List[TrainingExample]:
    """
    Generate multiple DIVERSE training examples using the same report as a template.
    
    Each example will be a completely different fake company with unique:
    - Company name and industry
    - Financial data
    - Competitive landscape
    - Investment verdict
    
    Args:
        example_report_path: Path to example report (used as structure template only)
        num_examples: Number of diverse fake companies to generate
        client: AsyncOpenAI client instance
        temperature: Sampling temperature (high = more diversity)
        
    Returns:
        List of TrainingExample objects, each with a unique fake company
    """
    import asyncio
    
    tasks = [
        generate_training_example(
            example_report_path=example_report_path,
            temperature=temperature,
            client=client
        )
        for _ in range(num_examples)
    ]
    
    results = await tqdm_asyncio.gather(*tasks, desc="Generating training examples")
    return results

In [5]:
def save_training_data(
    examples: List[TrainingExample],
    output_path: str = "training_data.jsonl"
):
    """
    Save training examples to JSONL format for fine-tuning.
    
    Args:
        examples: List of TrainingExample objects
        output_path: Path to output JSONL file
    """
    with open(output_path, 'w', encoding="utf-8") as f:
        for example in examples:
            # Convert to dict and write as JSON line
            f.write(example.model_dump_json() + '\n')
    
    print(f"Saved {len(examples)} training examples to {output_path}")

In [ ]:
batch = await generate_batch(
    example_report_path="../data/output/Duolingo_2025-11-18_21-47-42.md",
    num_examples=5000,
    client=client,
    temperature=1.0
)

Generating training examples:   0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
# Save to file
save_training_data(batch, "../data/training_data_examples.jsonl")

Saved 10 training examples to ../data/training_data_examples.jsonl


Stats

In [ ]:
from collections import Counter

# Count verdict statistics
verdict_counts = Counter([example.verdict.proceed_signal.value for example in batch])
print("Verdict Distribution:")
for verdict, count in verdict_counts.most_common():
    print(f"  {verdict}: {count} ({count/len(batch)*100:.1f}%)")

Verdict Distribution:
  strong_no: 5 (50.0%)
  strong_yes: 3 (30.0%)
  questionable: 2 (20.0%)


Explore an example

In [ ]:
print(batch[1].report[:1000])

Veloxion Technologies – Research Memo
Date: June 1, 2026

I. Company Overview and Business Model

– Veloxion Technologies, founded in 2017 and headquartered in Austin, TX, develops and sells autonomous delivery drones aimed at last-mile logistics for urban e-commerce retailers.
– The company designs proprietary drone hardware and software, and offers a SaaS platform for fleet management, regulatory compliance, and delivery optimization.
– Revenue streams:
  • Hardware sales: drones and associated docking stations
  • Software subscriptions: licensing fees for fleet management and analytics
  • Maintenance and service contracts
– Customer base: primarily medium to large e-commerce retailers and third-party logistics (3PL) providers in North America.
– As of FY 2025 revenue: $48.4 million, growing from $10.2 million in 2023.
– Funding: Raised $180 million to date in venture capital; no profitability, running significant operating losses.

II. Market and Competition

– Market size: Estima

In [ ]:
print(batch[1].reasoning[:1000])

Okay, looking at Veloxion Technologies, an autonomous drone delivery startup serving last-mile e-commerce logistics, the opportunity is intriguing but the current fundamentals look very concerning. Let me break down the key areas.

First, competition and market dynamics are tough. The market for drone delivery is nascent and heavily regulated. Veloxion is behind well-funded competitors like AeroFleet with proven tech and broader enterprise relationships. Regulatory hurdles remain a significant bottleneck, and no company has scaled beyond pilot stages yet.

Next, customers are mainly small regional retailers or 3PLs with pilot programs that show poor reliability, low volumes, and no long-term contracts. That indicates weak customer validation and retention risks.

Financially, Veloxion is bleeding cash fast. Revenue is growing but very small at $48 million in 2025. Gross margin is a declining 14% in 2025, driven down by hardware cost overruns and poor software monetization. Operating ex

In [ ]:
batch[1].verdict.proceed_signal.value

'strong_no'

In [ ]:
batch[1].verdict.evidence_pro

'1. Innovative technology targeting a disruptive and high-growth market: autonomous last-mile drone delivery for e-commerce.\n2. Early revenue growth with a near fivefold increase from 2023 to 2025 ($10M to $48M).\n3. Proprietary hardware and software platform with potential for SaaS subscription revenue expansion.\n4. Some pilot customers and initial operational deployments showing proof-of-concept potential.'

In [ ]:
batch[1].verdict.evidence_contra

'1. Large and growing net losses with negative EBITDA (-$41.2M in 2025) and declining gross margins (down to 14%).\n2. Limited and low-quality customer base mostly comprising small pilots with no long-term contracts.\n3. Significant technical execution risks: drone reliability issues, short battery life, and immature software platform.\n4. Regulatory uncertainty and slow approvals limiting market access and expansion.\n5. High cash burn leading to a cash runway under 3 quarters at current spending levels.\n6. Strong competition from better-capitalized and more established players (AeroFleet, others).\n7. Lack of proven unit economics or path to profitability in near-to-medium term.'

In [ ]:
batch[1].verdict.similar_deals

"- SilverLake Capital's 2024 growth equity investment in AeroFleet Inc., a leading autonomous drone delivery startup in North America.\n- Insight Partners' 2023 strategic investment in SkyPath Robotics focusing on warehouse automation drones.\n- Tiger Global’s $150M Series D funding round for GlobalLogix in 2025, scaling last-mile robotic delivery with ground vehicles.\n- Sequoia Capital’s 2022 early-stage investment in HoverX, a drone startup pivoting from hardware to pure software fleet management platforms."

In [ ]:
batch[1].verdict.key_information_requested

'1. Detailed technical validation reports on hardware reliability and software platform capabilities.\n2. Regulatory pathway updates and application timelines, especially FAA approvals.\n3. Customer contract pipeline and churn metrics for existing pilots.\n4. Updated cash flow projections showing runway extension or new funding sources.\n5. Competitor benchmarking on costs, product performance, and customer acquisition.\n6. Roadmap for improving battery life, reducing production costs, and software integration enhancements.'